## Init session

In [ ]:
%load_ext autoreload
%autoreload 2

### Install dependencies

In [ ]:
!chmod +x install.sh
! ./install.sh > /dev/null 2>&1

### Import packages

In [ ]:
import os
import boto3
import subprocess

from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import torch

#print(torch.version.cuda)           
#print(torch.backends.cudnn.version()) 
#print(torch.cuda.is_available())  

from sklearn.model_selection import train_test_split

import model as ic

## Load & clean data

### Import annotations

In [ ]:
tab_raw = pd.read_csv(Path(".").joinpath("data").joinpath("images_annotated.csv"))

binary_columns = tab_raw.columns[2:]

#print(tab_raw)
#print(tab_raw.iloc[:, 2:len(tab_raw)].sum())

### Select sites

In [ ]:
sites_trainvaltest = ["carpathians", "french_alps", "stubai_valley", "vinschgau"]
sites_external = ["danube", "dovre", "sierra_nevada"]

tab_external = tab_raw[tab_raw["site"].isin(sites_external)].copy()
tab_raw = tab_raw[tab_raw["site"].isin(sites_trainvaltest)]

print(tab_raw.shape)
print(tab_external.shape)

### Rebalance categories

In [ ]:
n = 2000
rng = np.random.default_rng(42)           

In [ ]:
# ------------------------------------------------------------------ #
# Compute weights from imbalance in the original dataframe
# ------------------------------------------------------------------ #
# p = proportion of 1s; distance from 0.5 ranges from 0 (perfect balance)
# to 0.5 (all 0s or all 1s). We map it to a weight >= 1.
# weight = 1 + k * (|p - 0.5| / 0.5)  with k controlling the max weight.
k = 4  # max additional weight on top of the baseline 1
weights = {}
for col in binary_columns:
    p = tab_raw[col].mean()
    imbalance = abs(p - 0.5) / 0.5  # 0 = perfectly balanced, 1 = fully skewed
    weights[col] = 1 + k * imbalance
    # print(f"  {col}: proportion of 1s = {p:.3f}, weight = {weights[col]:.2f}")

In [ ]:
# ------------------------------------------------------------------ #
# Greedy balanced selection
# ------------------------------------------------------------------ #
seed = 42  
rng = np.random.default_rng(seed)
df_shuffled = tab_raw.sample(frac=1, random_state=seed).reset_index(drop=True)

selected_indices = []
counts = {col: {0: 0, 1: 0} for col in binary_columns}
target = n // 2

base_tolerance = 500   
ramp = 1000

values = df_shuffled[binary_columns].values

for idx, row_vals in enumerate(values):
    if len(selected_indices) >= n:
        break

    score = 0

    for j, col in enumerate(binary_columns):
        val = int(row_vals[j])
        w = weights[col]

        score += w * (
            (target - counts[col][val])
            - (target - counts[col][1 - val])
        )

    # -------- trade-off control --------
    progress = len(selected_indices) / n
    threshold = -(base_tolerance + progress * ramp)

    if score >= threshold:
        selected_indices.append(idx)

        for j, col in enumerate(binary_columns):
            counts[col][int(row_vals[j])] += 1

tab = df_shuffled.loc[selected_indices].reset_index(drop = True)

selected_set = set(selected_indices)
tab_holdout = df_shuffled.loc[~df_shuffled.index.isin(selected_set)].reset_index(drop=True)

pd.DataFrame(
    data={
        "Category": [col for col in binary_columns],
        "%": [tab[col].mean() * 100 for col in binary_columns],
        "Number": [sum(tab[col]) for col in binary_columns],
        "Total": len(tab),
    }
).sort_values("%", ascending = False)

In [ ]:
print(tab.shape)
print(tab_holdout.shape)
print(tab_external.shape)

## Train

### Split dataset

In [ ]:
tab_strat = tab.copy()
tab_strat["strat"] = ""
for col in tab.columns[2:]:
    tab_strat["strat"] += tab_strat[col].astype(str)

#print(tab_strat)

In [ ]:
trainval, test = train_test_split(tab_strat, test_size = 0.15, random_state = 42, stratify = tab_strat["strat"])
test = test.drop("strat", axis=1)

#print(trainval)
#print(test)

### Mini-grid search

In [ ]:
minigrid = False
if minigrid:
    
    backbones = ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]
    repetitions = [3, 4, 5] 
    learning_rates = [0.001, 0.0001, 0.00001]
    batch_sizes = [16, 32]
    
    for rep in repetitions:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
        
        for backbone in backbones:
            for lr in learning_rates:
                for bs in batch_sizes:
                    
                    ic.train_model(
                        train_data=train,
                        val_data=val,
                        batch_size=bs,
                        max_epochs=20,  
                        image_size=224,
                        run_owner="onyxia",
                        exp_name="minigrid",
                        backbone=backbone,
                        loss_name="bce",
                        loss_params={"alpha":0.25, "gamma":2},  
                        device=ic.get_device(),
                        checkpoints_n_saved=1,
                        learning_rate=lr,
                        early_stoper_patience=5,
                        early_stoper_min_delta=0.001,
                        use_lr_finder=False,
                        lr_scheduler_step=10,
                        lr_scheduler_gamma=0.85,
                        print_steps="print",
                        log_progress=False,
                        plot_loss=False,
                        num_workers=10,
                    )


### Export minigrid outputs

In [ ]:
os.makedirs("outputs", exist_ok=True)

runs =  mlflow.search_runs(search_all_experiments = True, experiment_names = ["minigrid"])

runs_df = runs[[
    "run_id",
    "params.backbone",
    "params.batch_size",
    "params.learning_rate"
]].copy()

runs_df.columns = ["run_id", "backbone", "batch_size", "learning_rate"]

all_data = []
for _, row in runs_df.iterrows():
    run_id = row["run_id"]
    
    try:
        local_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path="metrics/classification_report.csv"
        )
        
        df_report = pd.read_csv(local_path, sep=";")
        
        df_report["run_id"] = run_id
        df_report["backbone"] = row["backbone"]
        df_report["batch_size"] = row["batch_size"]
        df_report["learning_rate"] = row["learning_rate"]
        
        all_data.append(df_report)
        
    except Exception as e:
        print(f"Error for {run_id}: {e}")

final_df = pd.concat(all_data, ignore_index=True)

cols_order = ["run_id", "backbone", "batch_size", "learning_rate"]
final_df = final_df[
    cols_order + [col for col in final_df.columns if col not in cols_order]
]

final_df.to_csv("outputs/minigrid_results.csv", index=False)

### Train 50 models for hf_swt_t

In [ ]:
trainswt = False
if trainswt:
    
    for rep in [11,12,13,14,15]:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
        
        for _ in list(range(10)):
            ic.train_model(
                train_data = train,
                val_data = val,
                batch_size = 32,
                max_epochs = 100,
                image_size = 224,
                run_owner = "onyxia",
                exp_name = "trainswt" + str(rep),
                backbone = "hf_swt_t",
                loss_name = "bce",
                loss_params = {"alpha": 0.25, "gamma": 2},
                device = ic.get_device(),
                checkpoints_n_saved = 1,
                learning_rate = 0.0001,
                early_stoper_patience = 10,
                early_stoper_min_delta = 0.001,
                use_lr_finder = False,
                lr_scheduler_step = 10,
                lr_scheduler_gamma = 0.85,
                print_steps = "print",
                log_progress = False,
                plot_loss = False,
                num_workers = 10,
            )


### Export trainswt outputs

In [ ]:
all_data = []
for rep in [11,12,13,14,15]:
    
    exp = mlflow.get_experiment_by_name(f"trainswt{rep}")
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    
    for _, row in runs.iterrows():
        run_id = row["run_id"]
        
        try:
            local_path = mlflow.artifacts.download_artifacts(
                run_id=run_id,
                artifact_path="metrics/classification_report.csv"
            )
            
            df_report = pd.read_csv(local_path, sep=";")
            
            df_report["run_id"] = run_id
            df_report["rep"] = rep 
            
            all_data.append(df_report)
            
        except Exception as e:
            print(f"Error for {run_id}: {e}")

final_df = pd.concat(all_data, ignore_index=True)

cols_order = ["run_id", "rep"]
final_df = final_df[
    cols_order + [col for col in final_df.columns if col not in cols_order]
]

final_df.to_csv("outputs/trainswt_results.csv", index=False)

### Find best model

In [ ]:
df_weighted = final_df[final_df["labels"] == "weighted avg"].copy()

print(df_weighted.loc[df_weighted["f1-score"].idxmax()])

In [ ]:
experiments = mlflow.search_experiments()
experiment_ids = [e.experiment_id for e in experiments if e.name.startswith("trainswt")]

runs = mlflow.search_runs(experiment_ids=experiment_ids)

best = runs.sort_values("params.F1_weighted_avg", ascending=False).iloc[0]

print(best)

best_run_id = best.run_id

### Export training outputs

In [ ]:
client = mlflow.tracking.MlflowClient()

train_loss = client.get_metric_history(best_run_id, "training Loss")
val_loss   = client.get_metric_history(best_run_id, "validation Loss")

train_f1 = client.get_metric_history(best_run_id, "training F1")
val_f1   = client.get_metric_history(best_run_id, "validation F1")


train_loss_df = pd.DataFrame([
    {"epoch": m.step, "train_loss": m.value}
    for m in train_loss
])

val_loss_df = pd.DataFrame([
    {"epoch": m.step, "val_loss": m.value}
    for m in val_loss
])

train_f1_df = pd.DataFrame([
    {"epoch": m.step, "train_f1": m.value}
    for m in train_f1
])

val_f1_df = pd.DataFrame([
    {"epoch": m.step, "val_f1": m.value}
    for m in val_f1
])

df_epoch = (
    train_loss_df
    .merge(val_loss_df, on="epoch", how="outer")
    .merge(train_f1_df, on="epoch", how="outer")
    .merge(val_f1_df, on="epoch", how="outer")
    .sort_values("epoch")
)

df_epoch.to_csv("outputs/best_model_curves.csv", index=False)

## Validation

### Load best model

In [ ]:
model = mlflow.pytorch.load_model(
    f"runs:/{best_run_id}/model",
    map_location=torch.device(ic.get_device()),
)

### Export thresholds

In [ ]:
pd.DataFrame(model.thresholds).to_csv("outputs/thresholds.csv", index=False)

### Export train and val predictions

In [ ]:
train, val = train_test_split(trainval,test_size=0.18,stratify=trainval["strat"],random_state=11)

os.makedirs("outputs/train", exist_ok=True)
train = train.drop("strat", axis=1)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=train, train_mode=False)))
proba.to_csv("outputs/train/train_proba.csv", index=False)
trainout = model.get_val_data(dataset=ic.FldDataset(data=train, train_mode=False))
trainout["predictions_revue"].to_csv("outputs/train/train_prediction_revue.csv", index=False)
trainout["classification_report"].to_csv("outputs/train/train_classification_report.csv", index=False)

os.makedirs("outputs/val", exist_ok=True)
val = val.drop("strat", axis=1)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=val, train_mode=False)))
proba.to_csv("outputs/val/val_proba.csv", index=False)
valout = model.get_val_data(dataset=ic.FldDataset(data=val, train_mode=False))
valout["predictions_revue"].to_csv("outputs/val/val_prediction_revue.csv", index=False)
valout["classification_report"].to_csv("outputs/val/val_classification_report.csv", index=False)

### Export test predictions

In [ ]:
os.makedirs("outputs/test", exist_ok=True)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=test, train_mode=False)))
proba.to_csv("outputs/test/test_proba.csv", index=False)
testout = model.get_val_data(dataset=ic.FldDataset(data=test, train_mode=False))
testout["predictions_revue"].to_csv("outputs/test/test_prediction_revue.csv", index=False)
testout["classification_report"].to_csv("outputs/test/test_classification_report.csv", index=False)

### Export holdout predictions

In [ ]:
os.makedirs("outputs/holdout", exist_ok=True)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=tab_holdout, train_mode=False)))
proba.to_csv("outputs/holdout/holdout_proba.csv", index=False)
holdoutout = model.get_val_data(dataset=ic.FldDataset(data=tab_holdout, train_mode=False))
holdoutout["predictions_revue"].to_csv("outputs/holdout/holdout_prediction_revue.csv", index=False)
holdoutout["classification_report"].to_csv("outputs/holdout/holdout_classification_report.csv", index=False)

### Export external predictions

In [ ]:
os.makedirs("outputs/external", exist_ok=True)
proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=tab_external, train_mode=False)))
proba.to_csv("outputs/external/external_proba.csv", index=False)
externalout = model.get_val_data(dataset=ic.FldDataset(data=tab_external, train_mode=False))
externalout["predictions_revue"].to_csv("outputs/external/external_prediction_revue.csv", index=False)
externalout["classification_report"].to_csv("outputs/external/external_classification_report.csv", index=False)

### Export holdout/external predictions per site

In [ ]:
sites = ["carpathians", "danube", "dovre", "french_alps", "sierra_nevada", "stubai_valley", "vinschgau"]

for site in sites:
    out_dir = f"outputs/sites/{site}"
    os.makedirs(out_dir, exist_ok=True)

    if site in sites_trainvaltest:
        subset = tab_holdout[tab_holdout["site"] == site].reset_index(drop=True)
    else:
        subset = tab_external[tab_external["site"] == site].reset_index(drop=True)
    
    dataset = ic.FldDataset(data=subset, train_mode=False)

    # probabilités
    proba = pd.DataFrame(model.predict_propabilities(dataset=dataset))
    proba.to_csv(f"{out_dir}/{site}_proba.csv", index=False)

    # sortie modèle
    challengeout = model.get_val_data(dataset=dataset)

    challengeout["predictions_revue"].to_csv(
        f"{out_dir}/{site}_prediction_revue.csv",
        index=False
    )

    challengeout["classification_report"].to_csv(
        f"{out_dir}/{site}_classification_report.csv",
        index=False
    )

### Zip outputs

In [ ]:
subprocess.run(["zip", "-rq", "outputs.zip", "outputs"], check=True)